# Gold Layer – Fact Table & Materialized Views
Sources from `nestle_dev_silver.core.sales_transactions_scd2` (current rows only).

| Cell | Object | Type |
|------|--------|------|
| 2 | `nestle_dev_gold.mart` schema | DDL |
| 3 | `fact_sales` | Delta table (CTAS) |
| 4 | `mv_sales_by_region` | Materialized View |
| 5 | `mv_sales_by_channel` | Materialized View |
| 6 | `mv_sales_by_product` | Materialized View |
| 7 | Validation | Row counts |

In [0]:
-- Create Gold schema if it does not yet exist
CREATE SCHEMA IF NOT EXISTS nestle_dev_gold.mart
COMMENT 'Gold layer – business-ready fact tables and aggregated materialized views';

In [0]:
-- Gold fact table: one row per current sales transaction
CREATE OR REPLACE TABLE nestle_dev_gold.mart.fact_sales
CLUSTER BY AUTO
COMMENT 'Gold fact table – current sales transactions sourced from Silver SCD2'
AS
SELECT
  transaction_id,
  product_id,
  region,
  channel,
  customer_id,
  quantity,
  unit_price,
  amount,
  dbt_valid_from AS valid_from
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE;

In [0]:
-- Aggregated revenue metrics by region, auto-refreshed when Silver SCD2 changes
CREATE OR REPLACE MATERIALIZED VIEW nestle_dev_gold.mart.mv_sales_by_region
TRIGGER ON UPDATE
COMMENT 'Gold MV – sales metrics aggregated by region'
AS
SELECT
  region,
  COUNT(DISTINCT transaction_id)   AS transaction_count,
  COUNT(DISTINCT customer_id)      AS customer_count,
  SUM(quantity)                    AS total_quantity,
  SUM(amount)                      AS total_revenue,
  AVG(amount)                      AS avg_order_value
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
GROUP BY region;

In [0]:
-- Aggregated revenue metrics by sales channel, auto-refreshed when Silver SCD2 changes
CREATE OR REPLACE MATERIALIZED VIEW nestle_dev_gold.mart.mv_sales_by_channel
TRIGGER ON UPDATE
COMMENT 'Gold MV – sales metrics aggregated by channel'
AS
SELECT
  channel,
  COUNT(DISTINCT transaction_id)   AS transaction_count,
  COUNT(DISTINCT customer_id)      AS customer_count,
  SUM(quantity)                    AS total_quantity,
  SUM(amount)                      AS total_revenue,
  AVG(amount)                      AS avg_order_value
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
GROUP BY channel;

In [0]:
-- Aggregated revenue metrics by product, auto-refreshed when Silver SCD2 changes
CREATE OR REPLACE MATERIALIZED VIEW nestle_dev_gold.mart.mv_sales_by_product
TRIGGER ON UPDATE
COMMENT 'Gold MV – sales metrics aggregated by product'
AS
SELECT
  product_id,
  COUNT(DISTINCT transaction_id)   AS transaction_count,
  COUNT(DISTINCT customer_id)      AS customer_count,
  SUM(quantity)                    AS total_quantity,
  SUM(amount)                      AS total_revenue,
  AVG(unit_price)                  AS avg_unit_price
FROM nestle_dev_silver.core.sales_transactions_scd2
WHERE dbt_is_current = TRUE
GROUP BY product_id;

In [0]:
-- Validate all Gold objects are populated
SELECT 'fact_sales'           AS gold_object, COUNT(*) AS row_count FROM nestle_dev_gold.mart.fact_sales
UNION ALL
SELECT 'mv_sales_by_region',  COUNT(*) FROM nestle_dev_gold.mart.mv_sales_by_region
UNION ALL
SELECT 'mv_sales_by_channel', COUNT(*) FROM nestle_dev_gold.mart.mv_sales_by_channel
UNION ALL
SELECT 'mv_sales_by_product', COUNT(*) FROM nestle_dev_gold.mart.mv_sales_by_product
ORDER BY gold_object;